# Pipeline

> Run OCR + fix the markdown headings + describe images/figures

In [ ]:
#| default_exp pipeline

This module aims to fix and enrich markdown headings from OCR'd PDF files by:

1. Fixing heading hierarchy that was corrupted during OCR
2. Optionnaly, adding page numbers to headings for better navigation
3. Describing figures

In [ ]:
#| export
from fastcore.all import *
from mistocr.core import read_pgs
from re import sub, findall, MULTILINE
from pydantic import BaseModel
from lisette import *
from lisette.core import completion
from typing import Callable
import os
import json
import shutil
from asyncio import Semaphore, gather, sleep

In [ ]:
#| export
@delegates(add_img_descs)
async def pdf_to_md(pdf_path:str, dst:str, ocr_output:str=None, model:str='claude-sonnet-4-5', add_img_desc:bool=True, progress:bool=True, **kwargs):
    "Convert PDF to markdown with fixed headings and image descriptions"
    from mistocr.core import ocr_pdf
    ocr_dir = Path(ocr_output) if ocr_output else Path(pdf_path).with_suffix('')
    n_steps = 3 if add_img_desc else 2
    if progress: print(f"Step 1/{n_steps}: Running OCR on {pdf_path}...")
    ocr_pdf(pdf_path, ocr_dir)
    if progress: print(f"Step 2/{n_steps}: Fixing heading hierarchy...")
    fix_hdgs(ocr_dir, model=model)
    if add_img_desc:
        if progress: print(f"Step 3/{n_steps}: Adding image descriptions...")
        await add_img_descs(ocr_dir, dst=dst, model=model, progress=progress, **kwargs)
    elif dst and Path(dst) != ocr_dir:
        shutil.copytree(ocr_dir, dst, dirs_exist_ok=True)
    if progress: print("Done!")

## Heading Hierarchy

Functions for detecting and fixing markdown heading levels



OCR'd PDF files often have corrupted heading hierarchies - headings may jump levels incorrectly (e.g., from H1 to H4) or use inconsistent levels for sections at the same depth. This section provides tools to automatically detect and fix these issues using LLMs, while also optionally adding page numbers for easier navigation.

The first step is extracting all headings from a markdown document so we can analyze their structure.

In [ ]:
#| export
def get_hdgs(
    md:str # Markdown file string
    ) -> L: # L of strings
    "Return the markdown headings"
    # Sanitize removing '#' in python snippet if any
    md = sub(r'```[\s\S]*?```', '', md)
    return L(findall(r'^#{1,6} .+$', md, MULTILINE))



In [ ]:
#| export
def add_pg_hdgs(
    md:str, # Markdown file string, 
    n:int # Page number
    ) -> str: # Markdown file string
    "Add page number to all headings in page markdown"
    md = sub(r'```[\s\S]*?```', '', md)
    def repl(m): return m.group(0) + f' ... page {n}'
    return sub(r'^#{1,6} .+$', repl, md, flags=MULTILINE)

The `add_pg_hdgs` function serves two important purposes:

**1. Creating unique heading identifiers**

When fixing heading hierarchies across an entire document, we need a way to distinguish between headings that have the same text but appear in different locations. For example, a document might have multiple "Introduction" or "Conclusion" headings in different chapters. By appending the page number to each heading, we create unique identifiers that allow us to build a lookup table mapping each specific heading instance to its corrected version. This assumes (reasonably) that the same heading text won't appear twice on a single page.

**2. Providing spatial context for LLMs**

Adding page numbers gives LLMs valuable positional information when analyzing the document structure. The page number helps the model understand:
- Where a heading sits in the overall document flow
- The relative distance between sections
- Whether headings that seem related are actually close together or far apart

This spatial awareness can significantly improve the LLM's ability to infer the correct hierarchical relationships between headings, especially in long documents where similar section names might appear at different structural levels.

For instance:

In [ ]:
#| eval: false
pgs = read_pgs('files/test/md_all/resnet', join=False)
pg0,pg0_with = pgs[0][:500],add_pg_hdgs(pgs[0], n=1)[:500]
print('Before:\n' + 80*'-' + f'\n{pg0}\n\nAfter:\n' + 80*'-' + f'\n{pg0_with}')

Before:
--------------------------------------------------------------------------------
# Deep Residual Learning for Image Recognition 

Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun<br>Microsoft Research<br>\{kahe, v-xiangz, v-shren, jiansun\}@microsoft.com


#### Abstract

Deeper neural networks are more difficult to train. We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously. We explicitly reformulate the layers as learning residual functions with reference to the layer inputs, instead of learning unr

After:
--------------------------------------------------------------------------------
# Deep Residual Learning for Image Recognition  ... page 1

Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun<br>Microsoft Research<br>\{kahe, v-xiangz, v-shren, jiansun\}@microsoft.com


#### Abstract ... page 1

Deeper neural networks are more difficult to train. We present a residual learning framework to ease the train

In [ ]:
#| export
def read_pgs_pg(
    path:str # Path to the markdown file
    ) -> L: # List of markdown pages
    "Read all pages of a markdown file and add page numbers to all headings"
    pgs = read_pgs(path, join=False)
    return L([add_pg_hdgs(pg, n) for n, pg in enumerate(pgs, 1)]).concat()

In [ ]:
#| eval: false
pgs = read_pgs_pg('files/test/md_all/resnet')
hdgs = L([get_hdgs(pg) for pg in pgs]).concat()
hdgs

(#22) ['# Deep Residual Learning for Image Recognition  ... page 1','#### Abstract ... page 1','## 1. Introduction ... page 1','## 2. Related Work ... page 2','## 3. Deep Residual Learning ... page 3','### 3.1. Residual Learning ... page 3','### 3.2. Identity Mapping by Shortcuts ... page 3','### 3.3. Network Architectures ... page 3','### 3.4. Implementation ... page 4','## 4. Experiments ... page 4','### 4.1. ImageNet Classification ... page 4','### 4.2. CIFAR-10 and Analysis ... page 7','### 4.3. Object Detection on PASCAL and MS COCO ... page 8','## References ... page 9','## A. Object Detection Baselines ... page 10','## PASCAL VOC ... page 10','## MS COCO ... page 10','## B. Object Detection Improvements ... page 10','## MS COCO ... page 10','## PASCAL VOC ... page 11'...]

To make it easier for an LLM to reference specific headings when suggesting fixes, we format them with index numbers.

In [ ]:
#| export
def fmt_hdgs_idx(
    hdgs: list[str] # List of markdown headings
    ) -> str: # Formatted string with index
    "Format the headings with index"
    return '\n'.join(f"{i}. {h}" for i, h in enumerate(hdgs))


In [ ]:
#| eval: false
hdgs_fmt = fmt_hdgs_idx(hdgs)
print(hdgs_fmt)

0. # Deep Residual Learning for Image Recognition  ... page 1
1. #### Abstract ... page 1
2. ## 1. Introduction ... page 1
3. ## 2. Related Work ... page 2
4. ## 3. Deep Residual Learning ... page 3
5. ### 3.1. Residual Learning ... page 3
6. ### 3.2. Identity Mapping by Shortcuts ... page 3
7. ### 3.3. Network Architectures ... page 3
8. ### 3.4. Implementation ... page 4
9. ## 4. Experiments ... page 4
10. ### 4.1. ImageNet Classification ... page 4
11. ### 4.2. CIFAR-10 and Analysis ... page 7
12. ### 4.3. Object Detection on PASCAL and MS COCO ... page 8
13. ## References ... page 9
14. ## A. Object Detection Baselines ... page 10
15. ## PASCAL VOC ... page 10
16. ## MS COCO ... page 10
17. ## B. Object Detection Improvements ... page 10
18. ## MS COCO ... page 10
19. ## PASCAL VOC ... page 11
20. ## ImageNet Detection ... page 11
21. ## C. ImageNet Localization ... page 12


We use a Pydantic model to ensure the LLM returns corrections in a structured format - a dictionary mapping heading indices to their corrected versions.

In [ ]:
#| export
class HeadingCorrections(BaseModel):
    corrections: dict[int, str]  # index → corrected heading

This prompt instructs the LLM on what types of heading hierarchy errors to fix while preserving the document's intended structure. It focuses on three main issues: 

- level jumps that skip intermediate levels, 
- numbering inconsistencies where subsection depth doesn't match heading level, and
- ensures decreasing levels (moving back up the hierarchy) are preserved.

In [ ]:
#| export
prompt_fix_hdgs = """Fix markdown heading hierarchy errors while preserving the document's intended structure.

INPUT FORMAT: Each heading is prefixed with its index number (e.g., "0. # Title ... page 1")

RULES - Apply these fixes in order:

1. **Single H1 rule**: Documents must have exactly ONE # heading (typically the document title at the top)
   - If index 0 is already #, then all subsequent headings (index 1+) must be ## or deeper
   - If no H1 exists, the first major heading should be #, and all others ## or deeper
   - NO exceptions: appendices, references, and all sections are ## or deeper after the title

2. **Infer depth from numbering patterns**: If headings contain section numbers, deeper nesting means deeper heading level
   - Parent section (e.g., "1", "2", "A") should be shallower than child (e.g., "1.1", "2.a", "A.1")
   - Child section should be one # deeper than parent
   - Works with any numbering: "1/1.1/1.1.1", "A/A.1/A.1.a", "I/I.A/I.A.1", etc.

3. **Level jumps**: Headings can only increase by one # at a time when moving deeper
   - Wrong: ## Section → ##### Subsection
   - Fixed: ## Section → ### Subsection

4. **Decreasing levels is OK**: Moving back up the hierarchy (### to ##) is valid for new sections

OUTPUT: Return a Python dictionary mapping index to corrected heading (without the index prefix).
IMPORTANT: Preserve the " ... page N" suffix in all corrected headings.
Only include entries that need changes.

Headings to analyze:
{headings_list}
"""


#| export
Now we can use an LLM to automatically detect and fix these heading hierarchy issues. The function uses litellm (wrapped by the [Lisette package](https://lisette.answer.ai) to send the formatted headings to a language model along with our correction rules. The LLM analyzes the structure and returns only the headings that need fixing, mapped by their index numbers.

In [ ]:
#| export
def fix_hdg_hierarchy(
    hdgs: list[str], # List of markdown headings
    prompt: str=None, # Prompt to use
    model: str='claude-sonnet-4-5', # Model to use
    api_key: str=None # API key
    ) -> dict[int, str]: # Dictionary of index → corrected heading
    "Fix the heading hierarchy"
    if api_key is None: api_key = os.getenv('ANTHROPIC_API_KEY')
    if prompt is None: prompt = prompt_fix_hdgs
    prompt = prompt.format(headings_list=fmt_hdgs_idx(hdgs))
    r = completion(model=model, messages=[{"role": "user", "content": prompt}], response_format=HeadingCorrections, api_key=api_key)
    return json.loads(r.choices[0].message.content)['corrections']


In [ ]:
#| eval: false
fixes = fix_hdg_hierarchy(hdgs)
fixes

{'1': '## Abstract ... page 1',
 '13': '## References ... page 9',
 '14': '## Appendix A. Object Detection Baselines ... page 10',
 '15': '### PASCAL VOC ... page 10',
 '16': '### MS COCO ... page 10',
 '17': '## Appendix B. Object Detection Improvements ... page 10',
 '18': '### MS COCO ... page 10',
 '19': '### PASCAL VOC ... page 11',
 '20': '### ImageNet Detection ... page 11',
 '21': '## Appendix C. ImageNet Localization ... page 12'}

The corrections come back as string indices, but we need to map the actual heading text to its corrected version for easy replacement in the document.

In [ ]:
#| export
@delegates(fix_hdg_hierarchy)
def mk_fixes_lut(
    hdgs: list[str], # List of markdown headings
    model: str='claude-sonnet-4-5', # Model to use
    api_key: str=None, # API key
    **kwargs
    ) -> dict[str, str]: # Dictionary of old → new heading
    "Make a lookup table of fixes"
    if api_key is None: api_key = os.getenv('ANTHROPIC_API_KEY')
    fixes = fix_hdg_hierarchy(hdgs, model=model, api_key=api_key, **kwargs)
    return {hdgs[int(k)]:v for k,v in fixes.items()}

In [ ]:
#| eval: false
lut_fixes = mk_fixes_lut(hdgs)
lut_fixes

{'#### Abstract ... page 1': '## Abstract ... page 1',
 '## References ... page 9': '## References ... page 9',
 '## A. Object Detection Baselines ... page 10': '## Appendix A. Object Detection Baselines ... page 10',
 '## PASCAL VOC ... page 10': '### PASCAL VOC ... page 10',
 '## MS COCO ... page 10': '### MS COCO ... page 10',
 '## B. Object Detection Improvements ... page 10': '## Appendix B. Object Detection Improvements ... page 10',
 '## PASCAL VOC ... page 11': '### PASCAL VOC ... page 11',
 '## ImageNet Detection ... page 11': '### ImageNet Detection ... page 11',
 '## C. ImageNet Localization ... page 12': '## Appendix C. ImageNet Localization ... page 12'}

Now we can apply the fixes to individual pages. We optionally add page numbers to headings for easier navigation in the final document.

In [ ]:
#| export
def apply_hdg_fixes(
    p:str, # Page to fix
    lut_fixes: dict[str, str], # Lookup table of fixes
    ) -> str: # Page with fixes applied
    "Apply the fixes to the page"
    for old in get_hdgs(p): p = p.replace(old, lut_fixes.get(old, old))
    return p

In [ ]:
#| eval: false
pg_nb = 1
p = read_pgs_pg('files/test/md_all/resnet')[0]
print(apply_hdg_fixes(p, lut_fixes)[:300])

# Deep Residual Learning for Image Recognition  ... page 1

Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun<br>Microsoft Research<br>\{kahe, v-xiangz, v-shren, jiansun\}@microsoft.com


## Abstract ... page 1

Deeper neural networks are more difficult to train. We present a residual learning framewor


Finally, we tie everything together in a single function that processes an entire document directory, fixing all heading hierarchy issues and optionally adding page numbers.

In [ ]:
fix_hdgs('md', dst='md_fixed')

In [ ]:
#| export
@delegates(mk_fixes_lut)
def fix_hdgs(src:str, model:str='claude-sonnet-4-5', dst:str=None, img_folder:str='img', **kwargs):
    "Fix heading hierarchy in markdown document"
    src_path,dst_path = Path(src),Path(dst) if dst else Path(src)
    if dst_path != src_path: dst_path.mkdir(parents=True, exist_ok=True)
    src_imgs = src_path/img_folder
    if src_imgs.exists() and dst_path != src_path: shutil.copytree(src_imgs, dst_path/img_folder, dirs_exist_ok=True)
    pgs_with_pg = read_pgs_pg(src_path)
    lut = mk_fixes_lut(L([get_hdgs(pg) for pg in pgs_with_pg]).concat(), model, **kwargs)
    for i,p in enumerate(pgs_with_pg, 1): (dst_path/f'page_{i}.md').write_text(apply_hdg_fixes(p, lut))

In [ ]:
#| eval: false
fix_hdgs('files/test/md_all/resnet', dst='files/test/md_fixed/resnet')

In [ ]:
#| eval: false
!ls -R 'files/test/md_fixed/resnet'

files/test/md_fixed/resnet:
img	   page_10.md  page_12.md  page_3.md  page_5.md  page_7.md  page_9.md
page_1.md  page_11.md  page_2.md   page_4.md  page_6.md  page_8.md

files/test/md_fixed/resnet/img:
img-0.jpeg  img-2.jpeg	img-4.jpeg  img-6.jpeg
img-1.jpeg  img-3.jpeg	img-5.jpeg


In [ ]:
#| eval: false
md = read_pgs('files/test/md_fixed/resnet')
print(md[:500])

# Deep Residual Learning for Image Recognition  ... page 1

Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun<br>Microsoft Research<br>\{kahe, v-xiangz, v-shren, jiansun\}@microsoft.com


## Abstract ... page 1

Deeper neural networks are more difficult to train. We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously. We explicitly reformulate the layers as learning residual functions with reference to the layer inputs, ins


## Image Description

Tools for classifying and describing images in markdown documents

In [ ]:
md = read_pgs('files/test/md_fixed/resnet')
print(md[:500])

# Deep Residual Learning for Image Recognition  ... page 1

Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun<br>Microsoft Research<br>\{kahe, v-xiangz, v-shren, jiansun\}@microsoft.com


## Abstract ... page 1

Deeper neural networks are more difficult to train. We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously. We explicitly reformulate the layers as learning residual functions with reference to the layer inputs, ins


In [ ]:
imgs = Path('files/test/md_fixed/resnet/img').ls(file_exts='.jpeg')
imgs

(#7) [Path('files/test/md_fixed/resnet/img/img-5.jpeg'),Path('files/test/md_fixed/resnet/img/img-0.jpeg'),Path('files/test/md_fixed/resnet/img/img-1.jpeg'),Path('files/test/md_fixed/resnet/img/img-2.jpeg'),Path('files/test/md_fixed/resnet/img/img-3.jpeg'),Path('files/test/md_fixed/resnet/img/img-6.jpeg'),Path('files/test/md_fixed/resnet/img/img-4.jpeg')]

In [ ]:
#| export
class ImgDescription(BaseModel):
    is_informative: bool
    description: str

In [ ]:
#| export
describe_img_prompt = """Analyze this image from an academic/technical document.

Step 1: Determine if this image is informative for understanding the document content.
- Informative: charts, diagrams, tables, technical illustrations, experimental results, architectural diagrams
- Non-informative: logos, decorative images, generic photos, page backgrounds

Step 2: 
- If informative: Provide a detailed description including the type of visualization, key elements and their relationships, important data or patterns, and relevant technical details.
- If non-informative: Provide a brief label (e.g., "Company logo", "Decorative header image")

Return your response as JSON with 'is_informative' (boolean) and 'description' (string) fields."""

In [ ]:
#| export
async def describe_img(
    img_path: Path,  # Path to the image file
    model: str = 'claude-sonnet-4-5',  # Model to use
    prompt: str = describe_img_prompt  # Prompt for description
) -> ImgDescription:
    "Describe a single image using AsyncChat"
    chat = AsyncChat(model=model)
    r = await chat([img_path.read_bytes(), prompt], response_format=ImgDescription)
    return r

In [ ]:
#|eval: false
img = Path('files/test/md_fixed/resnet/img/img-0.jpeg')
r = await describe_img(img)
r

{"is_informative": true, "description": "This figure contains two side-by-side line graphs comparing the performance of 56-layer and 20-layer neural networks during training. \n\nLeft panel: Shows \"training error (%)\" on the y-axis (ranging from 0 to 20%) versus \"iter. (1e4)\" on the x-axis (ranging from 0 to 6). The 56-layer network (red line) maintains higher training error around 18-20% initially, then drops sharply around iteration 3e4 to stabilize around 6-8%. The 20-layer network (yellow/olive line) starts around 20% and decreases more gradually and consistently to reach approximately 2-3% by iteration 6e4.\n\nRight panel: Shows \"test error (%)\" on the y-axis (same scale) versus \"iter. (1e4)\" on the x-axis (same range). Both networks show similar patterns to their training curves but with the test errors remaining higher. The 56-layer network (red) stabilizes around 14-16% after its drop, while the 20-layer network (yellow/olive) decreases more smoothly to around 10-11%.\n\nThis visualization demonstrates a degradation problem where the deeper 56-layer network performs worse than the shallower 20-layer network, both in training and test error, which is counterintuitive to the expectation that deeper networks should perform better."}

<details>

- id: `chatcmpl-36370183-252b-4832-8e2c-911fbd48a4b5`
- model: `claude-sonnet-4-5-20250929`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=365, prompt_tokens=1074, total_tokens=1439, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None), cache_creation_input_tokens=0, cache_read_input_tokens=0)`

</details>

In [ ]:
#| export
async def limit(semaphore, coro, delay=None):
    async with semaphore:
        r = await coro
        if delay: await sleep(delay)
        return r

In [ ]:
#|eval: false
img = Path('files/test/md_fixed/resnet/img/img-0.jpeg')
r = await limit(Semaphore(2), describe_img(img), delay=1)
r

{"is_informative": true, "description": "This figure contains two side-by-side line graphs comparing the performance of 56-layer and 20-layer neural networks during training. \n\nLeft panel: Shows \"training error (%)\" on the y-axis (ranging from 0 to 20%) versus \"iter. (1e4)\" on the x-axis (ranging from 0 to 6). The 56-layer network (red line) maintains higher training error around 18-20% initially, then drops sharply around iteration 3e4 to stabilize around 6-8%. The 20-layer network (yellow/olive line) starts around 20% and decreases more gradually and consistently to reach approximately 2-3% by iteration 6e4.\n\nRight panel: Shows \"test error (%)\" on the y-axis (same scale) versus \"iter. (1e4)\" on the x-axis (same range). Both networks show similar patterns to their training curves but with the test errors remaining higher. The 56-layer network (red) stabilizes around 14-16% after its drop, while the 20-layer network (yellow/olive) decreases more smoothly to around 10-11%.\n\nThis visualization demonstrates a degradation problem where the deeper 56-layer network performs worse than the shallower 20-layer network, both in training and test error, which is counterintuitive to the expectation that deeper networks should perform better."}

<details>

- id: `chatcmpl-02d90271-56b5-42ac-bbde-fd9ce3a7bb2e`
- model: `claude-sonnet-4-5-20250929`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=365, prompt_tokens=1074, total_tokens=1439, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None), cache_creation_input_tokens=0, cache_read_input_tokens=0)`

</details>

In [ ]:
#| export
def parse_r(result):
    "Parse Lisette's response"
    return json.loads(result.choices[0].message.content)

In [ ]:
#| export
async def describe_imgs(
    imgs: list[Path],
    model: str = 'claude-sonnet-4-5',
    prompt: str = describe_img_prompt,
    semaphore: int = 2,
    delay: float = 1
) -> dict[str, dict]:
    "Describe multiple images in parallel and return dict mapping filename to description"
    sem = Semaphore(semaphore)
    results = await gather(*[limit(sem, describe_img(img, model, prompt), delay) for img in imgs])
    return {img.name: parse_r(r) for img, r in zip(imgs, results)}

In [ ]:
#| eval: false
descs = await describe_imgs(imgs[:2])
descs

{'img-5.jpeg': {'is_informative': True,
  'description': 'This figure contains three line graphs showing training error rates over iterations for different neural network architectures. \n\nLeft panel: Shows error rates for "plain" networks (plain-20, plain-32, plain-44, plain-56) plotted over approximately 6×10^4 iterations. The curves show different convergence behaviors, with the 56-layer and 20-layer networks labeled specifically. Error rates range from 0-20%.\n\nMiddle panel: Displays error rates for ResNet architectures (ResNet-20, ResNet-32, ResNet-44, ResNet-56, ResNet-110) over the same iteration range. Multiple colored lines show convergence patterns, with 110-layer and 20-layer networks specifically labeled. The curves generally show smoother convergence compared to plain networks.\n\nRight panel: Shows a comparison between two residual networks (residual-110 in purple and residual-1202 in black) over iterations. Both curves stabilize around 5-8% error, with residual-1202 sh

In [ ]:
#| export
def save_img_descs(
    descs: dict, # Dictionary of image descriptions
    dst_fname: Path, # Path to save the JSON file
    ) -> None:    
    "Save image descriptions to JSON file"
    Path(dst_fname).write_text(json.dumps(descs, indent=2))

In [ ]:
#| eval: false
save_img_descs(descs, 'files/test/md_fixed/resnet/img_descriptions.json')
Path('files/test/md_fixed/resnet/img_descriptions.json').read_text()[:500]

'{\n  "img-5.jpeg": {\n    "is_informative": true,\n    "description": "This figure contains three line graphs showing training error rates over iterations for different neural network architectures. \\n\\nLeft panel: Shows error rates for \\"plain\\" networks (plain-20, plain-32, plain-44, plain-56) plotted over approximately 6\\u00d710^4 iterations. The curves show different convergence behaviors, with the 56-layer and 20-layer networks labeled specifically. Error rates range from 0-20%.\\n\\nMiddle pane'

In [ ]:
#| export
def add_descs_to_pg(pg:str, descs:dict) -> str:
    "Add AI-generated descriptions to images in page"
    for link in findall(r'\[\^\d+\]!\[[^\]]*\]\([^)]+\)', pg):
        fname = findall(r'\(([^)]+)\)', link)[0]
        if fname in descs: pg = pg.replace(link, f"{link}\nAI-generated image description:\n___\n{descs[fname]['description']}\n___")
    return pg

In [ ]:
pgs = read_pgs('files/test/md_fixed/resnet', join=False)
print(pgs[0][2000:3000])

3,16]$ on the challenging ImageNet dataset [36] all exploit "very deep" [41] models, with a depth of sixteen [41] to thirty [16]. Many other nontrivial visual recognition tasks $[8,12,7,32,27]$ have also

[^0]![img-0.jpeg](img-0.jpeg)

Figure 1. Training error (left) and test error (right) on CIFAR-10 with 20-layer and 56-layer "plain" networks. The deeper network has higher training error, and thus test error. Similar phenomena on ImageNet is presented in Fig. 4.
greatly benefited from very deep models.
Driven by the significance of depth, a question arises: Is learning better networks as easy as stacking more layers? An obstacle to answering this question was the notorious problem of vanishing/exploding gradients [1, 9], which hamper convergence from the beginning. This problem, however, has been largely addressed by normalized initialization $[23,9,37,13]$ and intermediate normalization layers [16], which enable networks with tens of layers to start converging for stochastic gradien

In [ ]:
descs = json.loads(Path('files/test/md_fixed/resnet/img_descriptions.json').read_text())
new_pg = add_descs_to_pg(pgs[0], descs)
print(new_pg[2000:4000])

3,16]$ on the challenging ImageNet dataset [36] all exploit "very deep" [41] models, with a depth of sixteen [41] to thirty [16]. Many other nontrivial visual recognition tasks $[8,12,7,32,27]$ have also

[^0]![img-0.jpeg](img-0.jpeg)
AI-generated image description:
___
This figure contains two side-by-side line graphs comparing the performance of 56-layer and 20-layer neural networks during training. 

Left panel: Shows "training error (%)" on the y-axis (ranging from 0 to 20%) versus "iter. (1e4)" on the x-axis (ranging from 0 to 6). The 56-layer network (red line) maintains higher training error around 18-20% initially, then drops sharply around iteration 3e4 to stabilize around 6-8%. The 20-layer network (yellow/olive line) starts around 20% and decreases more gradually and consistently to reach approximately 2-3% by iteration 6e4.

Right panel: Shows "test error (%)" on the y-axis (same scale) versus "iter. (1e4)" on the x-axis (same range). Both networks show similar patterns to 

In [ ]:
#| export
def add_descs_to_pgs(pgs:list, descs:dict) -> list:
    "Add AI-generated descriptions to images in all pages"
    return [add_img_desc_pg(pg, descs) for pg in pgs]

In [ ]:
#| eval: false
enriched_pgs = add_descs_to_pgs(pgs, descs)
print(enriched_pgs[0][2000:3000])

3,16]$ on the challenging ImageNet dataset [36] all exploit "very deep" [41] models, with a depth of sixteen [41] to thirty [16]. Many other nontrivial visual recognition tasks $[8,12,7,32,27]$ have also

[^0]![img-0.jpeg](img-0.jpeg)
AI-generated image description:
___
This figure contains two side-by-side line graphs comparing the performance of 56-layer and 20-layer neural networks during training. 

Left panel: Shows "training error (%)" on the y-axis (ranging from 0 to 20%) versus "iter. (1e4)" on the x-axis (ranging from 0 to 6). The 56-layer network (red line) maintains higher training error around 18-20% initially, then drops sharply around iteration 3e4 to stabilize around 6-8%. The 20-layer network (yellow/olive line) starts around 20% and decreases more gradually and consistently to reach approximately 2-3% by iteration 6e4.

Right panel: Shows "test error (%)" on the y-axis (same scale) versus "iter. (1e4)" on the x-axis (same range). Both networks show similar patterns to 

In [ ]:
#| export
async def add_img_descs(src:str, dst:str=None, model:str='claude-sonnet-4-5', img_folder:str='img', semaphore:int=2, delay:float=1, force:bool=False, progress:bool=True):
    "Describe images and add descriptions to markdown pages"
    src_path,dst_path = Path(src),Path(dst) if dst else Path(src)
    if dst_path != src_path: dst_path.mkdir(parents=True, exist_ok=True)
    desc_file = src_path/'img_descriptions.json'
    if desc_file.exists() and not force:
        if progress: print(f"Loading existing descriptions from {desc_file}")
        descs = json.loads(desc_file.read_text())
    else:
        imgs = (src_path/img_folder).ls(file_exts=['.jpeg', '.jpg', '.png'])
        if progress: print(f"Describing {len(imgs)} images...")
        descs = await describe_imgs(imgs, model, semaphore=semaphore, delay=delay)
        save_img_descs(descs, desc_file)
        if progress: print(f"Saved descriptions to {desc_file}")
    pgs = read_pgs(src_path, join=False)
    if progress: print(f"Adding descriptions to {len(pgs)} pages...")
    enriched = add_descs_to_pgs(pgs, descs)
    for i,pg in enumerate(enriched, 1): (dst_path/f'page_{i}.md').write_text(pg)
    if progress: print(f"Done! Enriched pages saved to {dst_path}")

In [ ]:
#|eval = false
await add_img_descs('files/test/md_fixed/resnet', force=True, progress=True)

Describing 7 images...


Saved descriptions to files/test/md_fixed/resnet/img_descriptions.json
Adding descriptions to 12 pages...
Done! Enriched pages saved to files/test/md_fixed/resnet


In [ ]:
pgs = read_pgs('files/test/md_fixed/resnet', join=False)
print(pgs[0][2000:4000])

3,16]$ on the challenging ImageNet dataset [36] all exploit "very deep" [41] models, with a depth of sixteen [41] to thirty [16]. Many other nontrivial visual recognition tasks $[8,12,7,32,27]$ have also

[^0]![img-0.jpeg](img-0.jpeg)
AI-generated image description:
___
This figure contains two side-by-side line graphs comparing the performance of 56-layer and 20-layer neural networks during training. 

Left panel: Shows "training error (%)" on the y-axis (ranging from 0 to 20%) versus "iter. (1e4)" on the x-axis (ranging from 0 to 6). The 56-layer network (red line) maintains higher training error around 18-20% initially, then drops sharply around iteration 3e4 to stabilize around 6-8%. The 20-layer network (yellow/olive line) starts around 20% and decreases more gradually and consistently to reach approximately 2-3% by iteration 6e4.

Right panel: Shows "test error (%)" on the y-axis (same scale) versus "iter. (1e4)" on the x-axis (same range). Both networks show similar patterns to 

## Global orchestrator
